# Step 2E: Define "Success" and Scope

This notebook establishes the formal definition of the target variable and applies the final dataset scope for the **Drama Success Copilot**.

### Formal Decisions Applied:
1. **Target Formulation (Option C - Relative MDL Audience Reach):**
   - Target is defined as the percentile rank of `watchers` within each `(country, release_year)` cohort.
   - It is binned into three decision tiers:
     - **High:** $\ge 75	ext{th}$ percentile *(Top quartile of reach within that country and year)*
     - **Medium:** $25	ext{th} - 75	ext{th}$ percentile *(Middle 50%)*
     - **Low:** $< 25	ext{th}$ percentile *(Bottom quartile)*
   - **Crucial Definition Note:** This target measures **relative audience reach on MyDramaList**, not absolute global commercial profit or critical quality.
2. **Scope Filter (Minimum 6 Episodes):**
   - The scope filter (`episodes >= 6`) is applied **first**, removing short-run titles, web shorts, and specials.
   - The 18 rows with missing `episodes` are dropped because they cannot satisfy the minimum threshold.
3. **Small Cohort Exclusion Rule (< 50 Dramas):**
   - Any `(country, release_year)` cohort containing fewer than 50 dramas after the scope filter is excluded from labeling and modeling to prevent noisy percentile thresholds.
4. **Complete Information Subset Defined:**
   - A `complete_info` indicator is created (`director`, `screenwriter`, `original_network`, `cast_summary` all not `"Unknown"`).
   - Models in Milestone 4 will be evaluated on both the full test set and this verified subset.


In [1]:
import numpy as np
import pandas as pd

# Load cleaned dataset from Step 2C (read-only input)
df_raw = pd.read_csv('../data/processed/dramas_clean.csv')
print(f"Loaded raw cleaned dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")


Loaded raw cleaned dataset: 10,400 rows, 30 columns


## 1. Behavior of Watchers and Ratings

We examine:
1. The distribution of raw `watchers` and $\log(1 + 	ext{watchers})$.
2. Median watchers across release years and countries.
3. Correlation between $\log(	ext{watchers})$ and `rating` for rated titles.


In [2]:
df_raw['log_watchers'] = np.log1p(df_raw['watchers'])

watch_summary = pd.DataFrame({
    'Raw Watchers': df_raw['watchers'].describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]),
    'Log(1 + Watchers)': df_raw['log_watchers'].describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
})
print("=== WATCHERS DISTRIBUTION SUMMARY ===")
print(watch_summary.round(2).to_string())


=== WATCHERS DISTRIBUTION SUMMARY ===
       Raw Watchers  Log(1 + Watchers)
count      10400.00           10400.00
mean        5862.21               6.00
std        19524.85               2.50
min            0.00               0.00
5%             7.00               2.08
10%           16.00               2.83
25%           68.00               4.23
50%          346.00               5.85
75%         2266.50               7.73
90%        13724.70               9.53
95%        30229.70              10.32
99%        98165.92              11.49
max       327227.00              12.70


In [3]:
# Median watchers by country and release year
pivot_watchers = df_raw.pivot_table(index='release_year', columns='country', values='watchers', aggfunc='median')
print("=== MEDIAN WATCHERS BY COUNTRY AND RELEASE YEAR ===")
print(pivot_watchers.to_string())

# Correlation between log watchers and rating (rated dramas only)
rated_mask = (df_raw['is_unranked'] == 0) & (df_raw['rating'].notna())
corr_logw_r = df_raw.loc[rated_mask, 'log_watchers'].corr(df_raw.loc[rated_mask, 'rating'])
print(f"\nRated dramas count:   {rated_mask.sum():,} ({rated_mask.mean()*100:.1f}%)")
print(f"Unrated dramas count: {(~rated_mask).sum():,} ({(~rated_mask).mean()*100:.1f}%)")
print(f"Pearson correlation [log(watchers), rating] (rated only): {corr_logw_r:.4f}")


=== MEDIAN WATCHERS BY COUNTRY AND RELEASE YEAR ===
country       China  Japan  South Korea  Thailand
release_year                                     
2015           72.5   98.0       1975.5     120.0
2016          144.0  103.0       1854.0     100.0
2017          312.0  123.0       1184.0     247.5
2018          392.0  115.0       1253.5     201.0
2019          797.0    NaN        961.0     307.5
2020            NaN    NaN       1020.0     478.0
2021          348.0  216.0       1463.0     508.0
2022          439.5  195.5       2034.0    1106.0
2023            NaN  171.0       5346.5    1030.0
2024            NaN    NaN       2118.0     669.5
2025            NaN    NaN       1613.0     346.0

Rated dramas count:   6,045 (58.1%)
Unrated dramas count: 4,355 (41.9%)
Pearson correlation [log(watchers), rating] (rated only): 0.5074


### What These Patterns Mean for Defining Success:
1. **Extreme Positive Skew:**
   - Raw watchers spans from 0 to 327,227 with a median of only 346 and a mean of 5,862.
   - Log transformation yields a symmetric distribution suitable for percentile ranking.
2. **Temporal Accumulation:**
   - Older dramas (2015–2017) have had up to 10 years to collect watchers, whereas newer dramas have had less time.
   - Normalizing watchers within each release-year cohort controls for this discrepancy.
3. **Country Platform Disparity:**
   - South Korean titles achieve vastly higher median watchers on MyDramaList than Japanese, Chinese, or Thai titles across all years.
   - Evaluating dramas relative to their country-year cohort ensures a Japanese drama is benchmarked against its Japanese peers, preventing Korean titles from dominating the upper tier.
4. **Survivor Bias in Ratings:**
   - 4,355 dramas (41.9%) have no published rating because they failed to meet MDL's vote threshold. Dropping them would introduce massive survivorship bias. Watchers, by contrast, is available for 100% of titles.


## 2. Scope Filter & Small-Cohort Exclusions

We now execute the approved scope decisions:
1. **Filter:** Keep only dramas with `episodes >= 6` (this also drops the 18 rows with missing `episodes`).
2. **Identify Small Cohorts:** Calculate cohort counts by `(country, release_year)` and flag any cohort with $N < 50$.
3. **Exclude Small Cohorts:** Exclude those specific cohorts from labeling and modeling.


In [4]:
# 1. Apply minimum 6 episodes scope filter
df_scoped = df_raw[df_raw['episodes'] >= 6].copy()
dropped_episodes = len(df_raw) - len(df_scoped)
print(f"Original dramas:                 {len(df_raw):,}")
print(f"Dramas dropped (< 6 eps or null): {dropped_episodes:,}")
print(f"Dramas after scope filter:       {len(df_scoped):,}")

# 2. Check cohort counts
cohort_sizes = df_scoped.groupby(['country', 'release_year']).size().reset_index(name='count')

# Identify small cohorts (< 50 titles)
small_cohorts = cohort_sizes[cohort_sizes['count'] < 50].copy()
print("\n=== EXCLUDED SMALL COHORTS (< 50 DRAMAS) ===")
print(small_cohorts.to_string(index=False))
total_excluded_small = small_cohorts['count'].sum()
print(f"Total dramas removed by small-cohort rule: {total_excluded_small:,}")

# 3. Filter out small cohorts
valid_cohort_keys = cohort_sizes[cohort_sizes['count'] >= 50].set_index(['country', 'release_year']).index
df_labeled = df_scoped[df_scoped.set_index(['country', 'release_year']).index.isin(valid_cohort_keys)].copy()

print(f"\nFinal labeled dataset size: {len(df_labeled):,} dramas")


Original dramas:                 10,400
Dramas dropped (< 6 eps or null): 2,784
Dramas after scope filter:       7,616

=== EXCLUDED SMALL COHORTS (< 50 DRAMAS) ===
    country  release_year  count
      China          2019     13
      Japan          2018     47
South Korea          2020     21
Total dramas removed by small-cohort rule: 81

Final labeled dataset size: 7,535 dramas


In [5]:
# Pivot table of remaining cohort sizes
pivot_remaining = df_labeled.pivot_table(index='release_year', columns='country', values='name', aggfunc='count', fill_value=0)
print("=== REMAINING COHORT SIZES BY COUNTRY AND YEAR ===")
print(pivot_remaining.to_string())

# Flag cohorts between 50 and 100 dramas
c_counts = df_labeled.groupby(['country', 'release_year']).size().reset_index(name='cohort_size')
cohorts_50_100 = c_counts[(c_counts['cohort_size'] >= 50) & (c_counts['cohort_size'] <= 100)]
print("\n=== COHORTS BETWEEN 50 AND 100 DRAMAS ===")
if len(cohorts_50_100) == 0:
    print("None. All retained cohorts have at least 118 dramas.")
else:
    print(cohorts_50_100.to_string(index=False))


=== REMAINING COHORT SIZES BY COUNTRY AND YEAR ===
country       China  Japan  South Korea  Thailand
release_year                                     
2015            274    153          139       138
2016            310    178          166       167
2017            320    194          194       168
2018            330      0          209       172
2019              0      0          230       178
2020              0      0            0       118
2021            692    277          225       178
2022            532    299          222       206
2023              0    153          177       201
2024              0      0          233       216
2025              0      0          119       167

=== COHORTS BETWEEN 50 AND 100 DRAMAS ===
None. All retained cohorts have at least 118 dramas.


## 3. Target Calculation (Option C)

Within each retained `(country, release_year)` cohort:
1. `cohort_size`: Number of dramas in that country-year cell.
2. `target_pct`: Percentile rank of `watchers` within the cell (`rank(pct=True)`).
3. `target_tier`:
   - **High:** `target_pct >= 0.75`
   - **Medium:** `0.25 <= target_pct < 0.75`
   - **Low:** `target_pct < 0.25`

*Note on Class Balance:* Because the tier thresholds are explicitly set at the 25th and 75th percentiles of each cohort, the 25/50/25 class balance holds **by construction**.


In [6]:
# Compute cohort size and percentile rank
df_labeled['cohort_size'] = df_labeled.groupby(['country', 'release_year'])['watchers'].transform('count')
df_labeled['target_pct'] = df_labeled.groupby(['country', 'release_year'])['watchers'].rank(pct=True)

# Assign decision tiers
def assign_tier(p):
    if p >= 0.75:
        return 'High'
    elif p >= 0.25:
        return 'Medium'
    else:
        return 'Low'

df_labeled['target_tier'] = df_labeled['target_pct'].apply(assign_tier)

print("=== OVERALL TIER DISTRIBUTION ===")
tier_summary = pd.DataFrame({
    'Count': df_labeled['target_tier'].value_counts()[['High', 'Medium', 'Low']],
    'Percentage': (df_labeled['target_tier'].value_counts(normalize=True)[['High', 'Medium', 'Low']] * 100).round(2)
})
print(tier_summary.to_string())


=== OVERALL TIER DISTRIBUTION ===
             Count  Percentage
target_tier                   
High          1905       25.28
Medium        3765       49.97
Low           1865       24.75


In [7]:
# Per-country tier mix
c_table = pd.crosstab(df_labeled['country'], df_labeled['target_tier'], normalize='index')[['High', 'Medium', 'Low']] * 100
print("=== PER-COUNTRY TIER DISTRIBUTION (%) ===")
print(c_table.round(1).to_string())

# Train (<= 2021) vs Test (>= 2022) balance
train_mask = (df_labeled['release_year'] <= 2021)
test_mask = (df_labeled['release_year'] >= 2022)

split_df = pd.DataFrame({
    'Train (<=2021)': (df_labeled.loc[train_mask, 'target_tier'].value_counts(normalize=True)[['High', 'Medium', 'Low']] * 100).round(1),
    'Test (>=2022)':  (df_labeled.loc[test_mask, 'target_tier'].value_counts(normalize=True)[['High', 'Medium', 'Low']] * 100).round(1)
})
print("\n=== TRAIN VS TEST TIER DISTRIBUTION (%) ===")
print(f"Train Count: {train_mask.sum():,} ({train_mask.mean()*100:.1f}%) | Test Count: {test_mask.sum():,} ({test_mask.mean()*100:.1f}%)")
print(split_df.to_string())


=== PER-COUNTRY TIER DISTRIBUTION (%) ===
target_tier  High  Medium   Low
country                        
China        25.2    50.1  24.7
Japan        25.3    49.9  24.8
South Korea  25.3    49.9  24.8
Thailand     25.4    49.9  24.7

=== TRAIN VS TEST TIER DISTRIBUTION (%) ===
Train Count: 5,010 (66.5%) | Test Count: 2,525 (33.5%)
             Train (<=2021)  Test (>=2022)
target_tier                               
High                   25.3           25.3
Medium                 50.0           49.9
Low                    24.7           24.8


## 4. Complete Information Subset Definition

We define an explicit boolean indicator:
`complete_info`: True when `director`, `screenwriter`, `original_network`, and `cast_summary` are all not `"Unknown"`.

### Purpose in Milestone 4:
To evaluate models on both the full realistic test set and this verified complete subset, preventing models from relying on page-completeness bias as an artificial shortcut.


In [8]:
df_labeled['complete_info'] = (
    (df_labeled['director'] != 'Unknown') &
    (df_labeled['screenwriter'] != 'Unknown') &
    (df_labeled['original_network'] != 'Unknown') &
    (df_labeled['cast_summary'] != 'Unknown')
)

ci_count = df_labeled['complete_info'].sum()
ci_pct = df_labeled['complete_info'].mean() * 100
print(f"Total complete_info titles: {ci_count:,} ({ci_pct:.1f}%)")

# Train vs Test complete_info
ci_train = df_labeled.loc[train_mask, 'complete_info'].sum()
ci_test = df_labeled.loc[test_mask, 'complete_info'].sum()
print(f"Train complete_info count: {ci_train:,} ({ci_train/train_mask.sum()*100:.1f}%) out of {train_mask.sum():,}")
print(f"Test complete_info count:  {ci_test:,} ({ci_test/test_mask.sum()*100:.1f}%) out of {test_mask.sum():,}")

# Breakdown by Country
c_ci = df_labeled.groupby('country')['complete_info'].agg(
    total='count',
    complete='sum',
    pct=lambda s: f"{s.mean()*100:.1f}%"
)
print("\n=== COMPLETE INFO BY COUNTRY ===")
print(c_ci.to_string())

# Tier mix inside complete_info vs incomplete
mix_ci = (df_labeled[df_labeled['complete_info']]['target_tier'].value_counts(normalize=True)[['High', 'Medium', 'Low']] * 100).round(1)
mix_inc = (df_labeled[~df_labeled['complete_info']]['target_tier'].value_counts(normalize=True)[['High', 'Medium', 'Low']] * 100).round(1)

tier_comp = pd.DataFrame({'Complete Info (N=3,044)': mix_ci, 'Incomplete Info (N=4,491)': mix_inc})
print("\n=== TIER MIX: COMPLETE INFO VS INCOMPLETE INFO (%) ===")
print(tier_comp.to_string())


Total complete_info titles: 3,044 (40.4%)
Train complete_info count: 2,022 (40.4%) out of 5,010
Test complete_info count:  1,022 (40.5%) out of 2,525

=== COMPLETE INFO BY COUNTRY ===
             total  complete    pct
country                            
China         2458       535  21.8%
Japan         1254       759  60.5%
South Korea   1914       934  48.8%
Thailand      1909       816  42.7%

=== TIER MIX: COMPLETE INFO VS INCOMPLETE INFO (%) ===
             Complete Info (N=3,044)  Incomplete Info (N=4,491)
target_tier                                                    
High                            40.4                       15.0
Medium                          49.9                       50.0
Low                              9.7                       35.0


## 5. Saving Labeled Dataset

We persist the labeled dataset to `data/processed/dramas_labeled.csv` (gitignored).
- Retains all raw and feature columns without dropping input features.
- Adds `target_pct`, `target_tier`, `cohort_size`, and `complete_info`.


In [9]:
out_path = '../data/processed/dramas_labeled.csv'
df_labeled.to_csv(out_path, index=False)
print(f"Saved {len(df_labeled):,} rows to {out_path}")
print("Columns added: target_pct, target_tier, cohort_size, complete_info")


Saved 7,535 rows to ../data/processed/dramas_labeled.csv
Columns added: target_pct, target_tier, cohort_size, complete_info
